# 03 — Named Entity Recognition (NER)
This notebook explores how ClauseGuard uses spaCy's pretrained NER model to automatically extract
key entities from a contract — parties, dates, monetary amounts, and locations.

**Production file:** `backend/ner.py`

In [ ]:
import sys
sys.path.append('..')

import spacy
from IPython.display import display, HTML

nlp = spacy.load("en_core_web_sm")
print("spaCy model:", nlp.meta['name'], "| version:", nlp.meta['version'])

## 1. Load and clean the contract

In [ ]:
from backend.extractor import extract_text
from backend.cleaner import clean_text

raw = extract_text("../tests/sample_contract.pdf")
cleaned = clean_text(raw)
print(f"Contract length: {len(cleaned)} characters")

## 2. Run spaCy NER and inspect raw entity labels

In [ ]:
doc = nlp(cleaned)

print(f"Total entities detected: {len(doc.ents)}\n")
print(f"{'Entity Text':<40} {'Label':<12} {'Explanation'}")
print("-" * 70)
for ent in doc.ents:
    print(f"{ent.text[:38]:<40} {ent.label_:<12} {spacy.explain(ent.label_)}")

## 3. Map spaCy labels to our simplified categories

In [ ]:
# spaCy has 18 entity types — we only care about 5
LABEL_MAP = {
    "PERSON": "parties",
    "ORG":    "parties",
    "DATE":   "dates",
    "MONEY":  "amounts",
    "GPE":    "locations",   # GPE = Geo-Political Entity
    "LOC":    "locations",
}

def extract_entities(text):
    doc = nlp(text)
    entities = {"parties": [], "dates": [], "amounts": [], "locations": []}
    for ent in doc.ents:
        category = LABEL_MAP.get(ent.label_)
        if category and ent.text not in entities[category]:
            entities[category].append(ent.text)
    return entities

entities = extract_entities(cleaned)
for category, items in entities.items():
    print(f"\n{category.upper()}:")
    for item in items:
        print(f"  • {item}")

## 4. Visualize entities inline with displacy

In [ ]:
# Render a portion of the contract with entity highlighting
excerpt = cleaned[:800]
doc_excerpt = nlp(excerpt)

colors = {
    "PERSON": "#ffd6cc",
    "ORG":    "#c8e6ff",
    "DATE":   "#d4f7d4",
    "MONEY":  "#fff3cc",
    "GPE":    "#e8ccff",
}

options = {"ents": list(colors.keys()), "colors": colors}
html = spacy.displacy.render(doc_excerpt, style="ent", jupyter=True, options=options)
display(HTML(html))

## 5. NER limitations on legal text

spaCy's `en_core_web_sm` was trained on news data, not legal contracts — some known quirks:

| Issue | Example | Fix |
|---|---|---|
| Company name misclassified | "ABC Corporation Address" → labelled as one ORG | Better contract-specific NER model |
| Markdown artifacts treated as MONEY | `## 1` flagged as amount | Clean markdown before NER |
| "Client" treated as location | spaCy confuses role labels | Domain-specific training data |

For production quality NER on contracts, a fine-tuned model (e.g. trained on CUAD dataset) would be needed.